# 13. REST API Demonstration

This notebook demonstrates how to interact with the SupportAI production REST API. It showcases requests and responses for all major endpoints:
- `GET /health` - Application readiness and component health
- `GET /version` - Semantic versioning and project information
- `POST /predict` - Intent classification, confidence scoring, and 3-tier routing
- `POST /retrieve` - Semantic vector search over historical resolved tickets
- `POST /explain` - Local prediction explainability using LIME attributions
- `GET /metrics` - Prometheus scrapable time-series metrics

We will run the server in the background and use Python `requests` library to query each endpoint.

In [1]:
import os
import subprocess
import time
import requests
import json

# Ensure server is run with TESTING=true to use the fast fallback model
env = os.environ.copy()
env["TESTING"] = "true"
env["PYTHONUNBUFFERED"] = "1"

print("Launching FastAPI server on http://localhost:8000...")
proc = subprocess.Popen(
    ["uvicorn", "src.api.app:app", "--host", "127.0.0.1", "--port", "8000"],
    env=env,
    cwd=".."
)

# Wait for initialization
time.sleep(15)
print("Server process spawned successfully. PID:", proc.pid)

Launching FastAPI server on http://localhost:8000...
Server process spawned successfully. PID: 17640


## 1. Health and Version Checks

In [2]:
base_url = "http://127.0.0.1:8000"

# GET /health
r_health = requests.get(f"{base_url}/health")
print("=== GET /health ===")
print("Status:", r_health.status_code)
print("Headers X-Request-ID:", r_health.headers.get("X-Request-ID"))
print("Body:", json.dumps(r_health.json(), indent=2))

print("\n")

# GET /version
r_version = requests.get(f"{base_url}/version")
print("=== GET /version ===")
print("Status:", r_version.status_code)
print("Headers X-Request-ID:", r_version.headers.get("X-Request-ID"))
print("Body:", json.dumps(r_version.json(), indent=2))

=== GET /health ===
Status: 200
Headers X-Request-ID: df1b6acd-7c17-411e-8691-192a7183a32b
Body: {
  "status": "healthy",
  "classifier_loaded": true,
  "retriever_loaded": true,
  "explainer_loaded": true
}


=== GET /version ===
Status: 200
Headers X-Request-ID: a61eee8d-2b7c-4c21-9ea8-d0f733d1b546
Body: {
  "project": "SupportAI",
  "version": "0.1.0",
  "description": "Production REST API for support ticket intelligence."
}


## 2. Intent Classification and Decision Routing

We query `POST /predict` using two queries:
1. A high-confidence query that auto-routes instantly.
2. A low-confidence query that defaults to human escalation or retrieval fallback depending on the routing thresholds.

In [3]:
# High Confidence Query
payload_high = {"text": "I forgot my passcode and cannot login"}
r_high = requests.post(f"{base_url}/predict", json=payload_high)
print("=== POST /predict (High Confidence) ===")
print("Status:", r_high.status_code)
print("Response:", json.dumps(r_high.json(), indent=2))

print("\n")

# Low/Mid Confidence Query
payload_low = {"text": "reset card pin number"}
r_low = requests.post(f"{base_url}/predict", json=payload_low)
print("=== POST /predict (Low/Mid Confidence) ===")
print("Status:", r_low.status_code)
print("Response:", json.dumps(r_low.json(), indent=2))

=== POST /predict (High Confidence) ===
Status: 200
Response: {
  "intent": "passcode_forgotten",
  "confidence": 0.9707167744636536,
  "route": "classifier",
  "retrieved_docs": [],
  "llm_used": false,
  "reply": "Automated routing to category: passcode_forgotten"
}


=== POST /predict (Low/Mid Confidence) ===
Status: 200
Response: {
  "intent": "unknown",
  "confidence": 0.42254090309143066,
  "route": "human_escalation",
  "retrieved_docs": [],
  "llm_used": false,
  "reply": "Escalated to human support review due to low confidence (0.4225)."
}


## 3. Semantic Retrieval and FAISS search

In [4]:
payload_retrieve = {"query": "reset pin number", "top_k": 2}
r_retrieve = requests.post(f"{base_url}/retrieve", json=payload_retrieve)
print("=== POST /retrieve ===")
print("Status:", r_retrieve.status_code)
print("Response:", json.dumps(r_retrieve.json(), indent=2))

=== POST /retrieve ===
Status: 200
Response: {
  "query": "reset pin number",
  "results": [
    {
      "rank": 1,
      "index": 1131,
      "score": 0.7412739992141724,
      "text": "how do i reset my pin, i can't seem to use my card?"
    },
    {
      "rank": 2,
      "index": 806,
      "score": 0.707464337348938,
      "text": "please tell me how to change my pin."
    }
  ]
}


## 4. Local Explainability (LIME)

In [5]:
payload_explain = {"text": "I forgot my passcode and cannot login", "num_features": 5, "num_samples": 20}
r_explain = requests.post(f"{base_url}/explain", json=payload_explain)
print("=== POST /explain ===")
print("Status:", r_explain.status_code)
resp_explain = r_explain.json()
# Truncate HTML visualization content for clean notebook output
resp_explain["explanation_html"] = resp_explain["explanation_html"][:120] + "... [truncated]"
print("Response:", json.dumps(resp_explain, indent=2))

=== POST /explain ===
Status: 200
Response: {
  "predicted_class": "passcode_forgotten",
  "predicted_probability": 0.9926396608352661,
  "attributions": [
    [
      "passcode",
      0.32063328839624666
    ],
    [
      "forgot",
      0.2659520693531092
    ],
    [
      "login",
      0.08775339879796613
    ],
    [
      "cannot",
      0.01896080816627574
    ],
    [
      "and",
      -0.0034520894758099476
    ]
  ],
  "explanation_html": "<html>\n        <meta http-equiv=\"content-type\" content=\"text/html; charset=UTF8\">\n        <head><script>var lime =\n/****... [truncated]"
}


## 5. Prometheus Observability Metrics

In [6]:
r_metrics = requests.get(f"{base_url}/metrics")
print("=== GET /metrics ===")
print("Status:", r_metrics.status_code)
print("First 10 lines of metrics:")
print("\n".join(r_metrics.text.splitlines()[:10]))

=== GET /metrics ===
Status: 200
First 10 lines of metrics:
# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 9835.0
python_gc_objects_collected_total{generation="1"} 1821.0
python_gc_objects_collected_total{generation="2"} 1865.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0


## Clean Up

In [7]:
print("Terminating background server...")
proc.terminate()
proc.wait()
print("Server terminated.")

Terminating background server...
Server terminated.
